# Module 4.4 — Wire RAG to the Decoder + Evaluate Grounding
**DeskMate SLM Curriculum · Phase 4 · Notebook 24**

End-to-end RAG: retrieve → assemble prompt → generate → cite → evaluate faithfulness.

| Stage | Component |
|---|---|
| Retrieve | `full_retrieve()` from Module 4.3 |
| Assemble | `build_rag_prompt()` with `[Source N]` labels |
| Generate | Fine-tuned Qwen2.5-1.5B QLoRA (Module 3.4) |
| Evaluate | Faithfulness (n-gram) + citation rate + relevance |

Read `4.4_rag_decoder.md` for full theory.

---

## Step 0 — Install

In [ ]:
%%capture
!pip install -q sentence-transformers==3.1.1 faiss-cpu==1.8.0 chromadb==0.5.5 \
               rank-bm25==0.2.2 transformers==4.44.0 peft==0.12.0 \
               bitsandbytes==0.43.3 torch==2.3.1 accelerate==0.34.0 \
               rouge-score==0.1.2

In [ ]:
import re, json, pathlib, pickle, random, os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from rouge_score import rouge_scorer as rs

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device  = 'cuda' if torch.cuda.is_available() else 'cpu'
HAS_GPU = device == 'cuda'
print(f'Device: {device}')

## Step 1 — Runtime & Paths

In [ ]:
try:
    import google.colab; RUNTIME = 'colab'
    PROJECT_ROOT = pathlib.Path('/content/slm')
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = 'kaggle'
        PROJECT_ROOT = pathlib.Path('/kaggle/working/slm')
    except ImportError:
        RUNTIME = 'local'
        PROJECT_ROOT = pathlib.Path('.').resolve()

DATA_PROC  = PROJECT_ROOT / 'data' / 'processed'
FAISS_DIR  = PROJECT_ROOT / 'data' / 'faiss'
CHROMA_DIR = PROJECT_ROOT / 'data' / 'chroma_db'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS    = PROJECT_ROOT / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
print(f'Runtime: {RUNTIME}')

## Step 2 — Load Retriever Stack (from Modules 4.2 & 4.3)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

# Chunk records
with open(DATA_PROC / 'chunk_records.jsonl') as f:
    chunk_records = [json.loads(l) for l in f]
print(f'Chunks: {len(chunk_records)}')

# FAISS index
faiss_index = faiss.read_index(str(FAISS_DIR / 'deskmate_faq.index'))
print(f'FAISS: {faiss_index.ntotal} vectors')

# Embedding model
BGE_PREFIX = 'Represent this sentence for searching relevant passages: '
embedder   = SentenceTransformer('BAAI/bge-small-en-v1.5')

# BM25
bm25_data = pickle.load(open(DATA_PROC / 'bm25_corpus.pkl', 'rb'))
bm25 = bm25_data['bm25']

# Reranker
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('Retriever stack loaded.')

In [ ]:
# Retriever pipeline (from Module 4.3)
def rrf(ranked_id_lists, k=60):
    scores = {}
    for ranked in ranked_id_lists:
        for rank, doc_id in enumerate(ranked):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

def full_retrieve(query, fields=None, k_final=3):
    product_filter = (fields or {}).get('product')
    # Dense
    q_emb = embedder.encode(BGE_PREFIX + query, normalize_embeddings=True)
    D, I  = faiss_index.search(np.array([q_emb], dtype='float32'), k=20)
    dense_hits = []
    for score, idx in zip(D[0], I[0]):
        if idx < 0: continue
        r = chunk_records[idx]
        if product_filter and r['product'] not in (product_filter, 'all'):
            continue
        dense_hits.append(r['id'])
    # BM25
    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_ids    = [chunk_records[i]['id'] for i in bm25_scores.argsort()[::-1][:20]]
    # RRF
    merged     = rrf([dense_hits, bm25_ids])
    id_to_chunk = {r['id']: r for r in chunk_records}
    candidates  = [id_to_chunk[cid] for cid in merged if cid in id_to_chunk][:20]
    # Rerank
    if candidates:
        pairs  = [(query, c['text']) for c in candidates]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
        return [c for _, c in ranked[:k_final]]
    return candidates[:k_final]

print('Retriever ready.')

## Step 3 — Load Fine-Tuned Decoder (Module 3.4)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME   = 'Qwen/Qwen2.5-1.5B'
ADAPTER_PATH = MODELS_DIR / 'deskmate-qlora-adapter'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
) if HAS_GPU else None

print(f'Loading {MODEL_NAME} ...')
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_cfg,
    torch_dtype=torch.bfloat16 if HAS_GPU else torch.float32,
    device_map='auto' if HAS_GPU else None,
    trust_remote_code=True,
)
if ADAPTER_PATH.exists():
    decoder = PeftModel.from_pretrained(base, str(ADAPTER_PATH))
    print('SFT adapter loaded.')
else:
    decoder = base
    print('No adapter found — using base model (run Module 3.4 first for best results).')
decoder.eval()
print('Decoder ready.')

## Step 4 — RAG Prompt Builder

In [ ]:
SYSTEM_RAG = (
    'You are DeskMate, a concise support assistant. '
    'Answer the ticket using ONLY the provided context passages. '
    'Cite each claim with [Source N] where N is the passage number. '
    'If the context does not contain the answer, respond with: '
    "'I don't have that information — please contact support@deskmate.com.'"
)

def build_rag_prompt(ticket, chunks, context=None):
    ctx_parts = []
    for i, chunk in enumerate(chunks, start=1):
        src = f"{chunk['source']} ({chunk['section']})"
        ctx_parts.append(f'[Source {i}] {src}\n{chunk["text"]}')
    context_block = '\n\n'.join(ctx_parts)
    user_content  = f'Context:\n{context_block}\n\nTicket: {ticket}'
    return [
        {'role': 'system', 'content': SYSTEM_RAG},
        {'role': 'user',   'content': user_content},
    ]

# Preview
demo_ticket = 'I was charged twice for my subscription last month.'
demo_chunks = full_retrieve(demo_ticket, k_final=2)
demo_msgs   = build_rag_prompt(demo_ticket, demo_chunks)
print('System:', demo_msgs[0]['content'][:80], '...')
print('\nUser (first 400 chars):', demo_msgs[1]['content'][:400], '...')

## Step 5 — RAG Generate Function

In [ ]:
def rag_generate(ticket, fields=None, k=3, max_new_tokens=150):
    chunks  = full_retrieve(ticket, fields=fields, k_final=k)
    messages = build_rag_prompt(ticket, chunks)
    prompt  = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs  = tokenizer(prompt, return_tensors='pt',
                        truncation=True, max_length=1024)
    inputs  = {k2: v.to(decoder.device if hasattr(decoder, 'device') else device)
               for k2, v in inputs.items()}
    with torch.no_grad():
        out = decoder.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id)
    new_toks = out[0][inputs['input_ids'].shape[1]:]
    reply    = tokenizer.decode(new_toks, skip_special_tokens=True).strip()
    return reply, chunks

def no_rag_generate(ticket, max_new_tokens=150):
    system = 'You are DeskMate, a concise support assistant. Respond in 2-4 sentences.'
    msgs   = [{'role':'system','content':system},
              {'role':'user',  'content':f'Ticket: {ticket}'}]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt',
                       truncation=True, max_length=512)
    inputs = {k2: v.to(decoder.device if hasattr(decoder,'device') else device)
              for k2, v in inputs.items()}
    with torch.no_grad():
        out = decoder.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id)
    new_toks = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_toks, skip_special_tokens=True).strip()

## Step 6 — Single Example End-to-End

In [ ]:
ticket = 'I was charged twice for my subscription last month. Can I get a refund?'
fields = {'product': 'all', 'intent': 'billing_dispute'}

rag_reply, rag_chunks = rag_generate(ticket, fields=fields)
base_reply = no_rag_generate(ticket)

print('=== Retrieved chunks ===')
for i, c in enumerate(rag_chunks, 1):
    print(f'[Source {i}] {c["source"]}')
    print(f'  {c["text"][:120]}...')
print()
print('=== No-RAG reply ===')
print(base_reply)
print()
print('=== RAG reply ===')
print(rag_reply)

## Step 7 — Citation Extractor

In [ ]:
def extract_citations(reply, n_sources):
    cited = set()
    for m in re.finditer(r'\[Source (\d+)\]', reply):
        n = int(m.group(1))
        if 1 <= n <= n_sources:
            cited.add(n)
    return sorted(cited)

cited = extract_citations(rag_reply, n_sources=len(rag_chunks))
print(f'Sources cited in RAG reply: {cited}')
print(f'Sources available         : {list(range(1, len(rag_chunks)+1))}')
if not cited:
    print('WARNING: model did not cite any sources — '
           'check grounding instruction or add RAG examples to SFT data.')

## Step 8 — Faithfulness Scorer (n-gram Entailment)

In [ ]:
def is_faithful_sentence(sentence, chunks, threshold=0.60):
    content_words = set(re.findall(r'\b\w{4,}\b', sentence.lower()))
    if not content_words:
        return True
    source_text  = ' '.join(c['text'] for c in chunks).lower()
    source_words = set(re.findall(r'\b\w{4,}\b', source_text))
    overlap = content_words & source_words
    return len(overlap) / len(content_words) >= threshold

def faithfulness_score(reply, chunks):
    sents = re.split(r'(?<=[.!?])\s+', reply.strip())
    sents = [s for s in sents if s.strip()]
    if not sents:
        return 1.0
    faithful = sum(is_faithful_sentence(s, chunks) for s in sents)
    return faithful / len(sents)

# Demo on single example
f_rag  = faithfulness_score(rag_reply,  rag_chunks)
f_base = faithfulness_score(base_reply, rag_chunks)  # base had no retrieval
print(f'Faithfulness (n-gram):')
print(f'  RAG reply   : {f_rag:.2f}')
print(f'  No-RAG reply: {f_base:.2f}  (no context was retrieved for base)')

## Step 9 — Load Gold Eval Set

In [ ]:
gold_path = DATA_PROC / 'gold.jsonl'
if gold_path.exists():
    gold_df = pd.read_json(gold_path, lines=True).head(50)
    if 'reference_reply' not in gold_df.columns:
        gold_df['reference_reply'] = gold_df.get('reply', '')
else:
    INTENTS = ['account_access','billing_dispute','technical_bug',
               'usage_question','outage_report']
    TICKETS = [
        ('I cannot log in — password reset email never arrived.',
         'account_access',
         'Please use the Forgot Password link on the login page. '
         'A reset email arrives within 2 minutes; check spam if not received.'),
        ('I was charged twice for the Pro plan.',
         'billing_dispute',
         'We have confirmed the duplicate charge and will refund it within 3 business days.'),
        ('CSV export gives a 500 error on large datasets.',
         'technical_bug',
         'This is a known issue for datasets over 50k rows. '
         'A fix ships in version 4.3.0. Workaround: filter to under 50k rows.'),
        ('How do I invite a team member to my workspace?',
         'usage_question',
         'Go to Settings > Team > Invite Member and enter their email. '
         'They receive an invitation within 5 minutes.'),
        ('Is DeskMate currently experiencing an outage?',
         'outage_report',
         'Check status.deskmate.io for real-time status. '
         'Subscribe to alerts for automatic notifications.'),
    ]
    rows = []
    for i in range(50):
        t, intent, ref = TICKETS[i % len(TICKETS)]
        rows.append({'ticket': t, 'intent': intent, 'reference_reply': ref})
    gold_df = pd.DataFrame(rows)

print(f'Gold eval set: {len(gold_df)} examples')

## Step 10 — Batch Generation (RAG vs No-RAG)

In [ ]:
rag_replies   = []
base_replies  = []
retrieved_chunks_all = []

for _, row in gold_df.iterrows():
    ticket = row.get('ticket', row.get('text', ''))
    fields = {'product': 'all', 'intent': row.get('intent', '')}
    rag_r, chunks = rag_generate(ticket, fields=fields)
    base_r        = no_rag_generate(ticket)
    rag_replies.append(rag_r)
    base_replies.append(base_r)
    retrieved_chunks_all.append(chunks)

print(f'Generated {len(rag_replies)} RAG replies and {len(base_replies)} no-RAG replies.')

## Step 11 — Faithfulness Scores (All Examples)

In [ ]:
rag_faith  = [faithfulness_score(r, c)
              for r, c in zip(rag_replies, retrieved_chunks_all)]
base_faith = [faithfulness_score(r, c)
              for r, c in zip(base_replies, retrieved_chunks_all)]

print(f'Faithfulness (n-gram):')
print(f'  RAG    : mean={np.mean(rag_faith):.3f}  '
         f'median={np.median(rag_faith):.3f}')
print(f'  No-RAG : mean={np.mean(base_faith):.3f}  '
         f'median={np.median(base_faith):.3f}')

# Distribution plot
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(base_faith, bins=15, alpha=0.6, label='No-RAG')
ax.hist(rag_faith,  bins=15, alpha=0.6, label='RAG')
ax.set_xlabel('Faithfulness score (n-gram)')
ax.set_ylabel('Count')
ax.set_title('Faithfulness Distribution — RAG vs No-RAG')
ax.legend(); plt.tight_layout(); plt.show()

## Step 12 — Citation Rate & Hallucination Check

In [ ]:
citation_counts = [
    len(extract_citations(r, n_sources=len(c)))
    for r, c in zip(rag_replies, retrieved_chunks_all)
]
citation_rate = sum(1 for n in citation_counts if n > 0) / len(citation_counts)

def flag_hallucinations(ticket, reply, chunks):
    source  = ticket + ' ' + ' '.join(c['text'] for c in chunks)
    s_ents  = set(re.findall(r'\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)*\b', source))
    r_ents  = re.findall(r'\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)*\b', reply)
    stop    = {'We','Our','Your','Please','Thank','The','I','You','This',
               'That','A','An','In','On','At','To','For','Of','It','Is',
               'Are','Was','Were','DeskMate','Pro','Source'}
    return [e for e in set(r_ents) if e not in s_ents and e not in stop]

rag_halluc  = [flag_hallucinations(
    gold_df.iloc[i].get('ticket',''), rag_replies[i], retrieved_chunks_all[i])
    for i in range(len(rag_replies))]
base_halluc = [flag_hallucinations(
    gold_df.iloc[i].get('ticket',''), base_replies[i], retrieved_chunks_all[i])
    for i in range(len(base_replies))]

rag_halluc_rate  = sum(1 for h in rag_halluc  if h) / len(rag_halluc)
base_halluc_rate = sum(1 for h in base_halluc if h) / len(base_halluc)

print(f'Citation rate (RAG)         : {citation_rate*100:.0f}%')
print(f'Hallucination rate (RAG)    : {rag_halluc_rate*100:.0f}%')
print(f'Hallucination rate (No-RAG) : {base_halluc_rate*100:.0f}%')

## Step 13 — ROUGE-L vs Reference

In [ ]:
rouge = rs.RougeScorer(['rougeL'], use_stemmer=True)
refs  = gold_df['reference_reply'].tolist()

rag_rouge  = [rouge.score(r, p)['rougeL'].fmeasure
              for r, p in zip(refs, rag_replies)]
base_rouge = [rouge.score(r, p)['rougeL'].fmeasure
              for r, p in zip(refs, base_replies)]

print(f'ROUGE-L: RAG={np.mean(rag_rouge):.3f}  No-RAG={np.mean(base_rouge):.3f}')

## Step 14 — Answer Relevance (LLM Judge, Optional)

In [ ]:
RUN_JUDGE = False   # Set True with ANTHROPIC_API_KEY in Colab Secrets

rag_relevance  = []
base_relevance = []

if RUN_JUDGE:
    import anthropic
    def get_api_key():
        key = os.environ.get('ANTHROPIC_API_KEY', '')
        if not key:
            try:
                from google.colab import userdata
                key = userdata.get('ANTHROPIC_API_KEY')
            except Exception: pass
        return key
    client = anthropic.Anthropic(api_key=get_api_key())
    PROMPT = (
        'Does this support reply directly address the customer ticket? '
        'Reply with a single integer 1-5 (1=off-topic, 5=fully answers).'
        '\n\nTicket: {ticket}\nReply: {reply}\nScore:'
    )
    N = min(20, len(gold_df))
    for i in range(N):
        ticket = gold_df.iloc[i].get('ticket','')
        for replies, scores_list in [(rag_replies, rag_relevance),
                                     (base_replies, base_relevance)]:
            resp = client.messages.create(
                model='claude-haiku-4-5-20251001', max_tokens=5,
                messages=[{'role':'user','content':
                           PROMPT.format(ticket=ticket, reply=replies[i])}])
            raw = resp.content[0].text.strip()
            try:
                scores_list.append(int(re.search(r'[1-5]', raw).group()))
            except Exception:
                scores_list.append(3)
    print(f'Relevance: RAG={np.mean(rag_relevance):.2f}  '
           f'No-RAG={np.mean(base_relevance):.2f}')
else:
    print('Relevance judge skipped (RUN_JUDGE=False).')

## Step 15 — Root Cause Analysis (Low-Faithfulness Replies)

In [ ]:
LOW_THRESHOLD = 0.70
low_faith = [
    (i, rag_faith[i], rag_replies[i], retrieved_chunks_all[i])
    for i in range(len(rag_replies))
    if rag_faith[i] < LOW_THRESHOLD
]
print(f'Low-faithfulness replies (< {LOW_THRESHOLD}): {len(low_faith)}')

for i, score, reply, chunks in low_faith[:3]:
    ticket = gold_df.iloc[i].get('ticket','')
    cited  = extract_citations(reply, len(chunks))
    print(f'\n--- Example {i} (faithfulness={score:.2f}) ---')
    print(f'Ticket : {ticket}')
    print(f'Reply  : {reply[:200]}...' if len(reply) > 200 else f'Reply: {reply}')
    print(f'Citations: {cited}')
    if not cited:
        print('ROOT CAUSE: model did not cite — grounding instruction too weak '
               'or citation format not in training distribution.')
    else:
        print('ROOT CAUSE: cited but still unfaithful — '
               'decoder blended memorised priors with context. '
               'Check if SFT data contained conflicting facts.')

## Step 16 — Faithfulness Scorecard

In [ ]:
rel_rag_str  = f'{np.mean(rag_relevance):.2f}'  if rag_relevance  else 'N/A'
rel_base_str = f'{np.mean(base_relevance):.2f}' if base_relevance else 'N/A'

rows = [
    ('Faithfulness (n-gram)',    f'{np.mean(base_faith):.3f}', f'{np.mean(rag_faith):.3f}'),
    ('Citation rate',            '0%',                          f'{citation_rate*100:.0f}%'),
    ('Answer relevance (1-5)',    rel_base_str,                  rel_rag_str),
    ('Hallucination rate',        f'{base_halluc_rate*100:.0f}%', f'{rag_halluc_rate*100:.0f}%'),
    ('ROUGE-L vs reference',      f'{np.mean(base_rouge):.3f}',  f'{np.mean(rag_rouge):.3f}'),
]

print(f'{"Metric":<30} {"No-RAG":>12} {"RAG":>12}')
print('-' * 56)
for metric, base_val, rag_val in rows:
    print(f'{metric:<30} {base_val:>12} {rag_val:>12}')

# Save
lines = [
    '# DeskMate RAG Faithfulness Scorecard\n',
    '| Metric | No-RAG | RAG |',
    '|---|---|---|',
]
for metric, base_val, rag_val in rows:
    lines.append(f'| {metric} | {base_val} | {rag_val} |')
lines += [
    '',
    '## Root Causes for Remaining Hallucinations\n',
    '1. Grounding instruction not strong enough (use ONLY)',
    '2. Context too long — attention diffusion past ~900 tokens',
    '3. Prompt template mismatch (context must be in user turn)',
    '4. SFT data conflicting with retrieved facts',
    '5. Citation format not in SFT training distribution',
]
sc_path = REPORTS / 'rag_scorecard.md'
sc_path.write_text('\n'.join(lines))
print(f'\nScorecard saved: {sc_path}')

## Checkpoint

> **Retrieval is perfect but answers still hallucinate — where do you look?**

Write your answer below.

In [ ]:
answer = """
[Write your answer here]
"""
print(answer)

## Deliverable Summary

| Artifact | Location |
|---|---|
| Faithfulness scorecard | `reports/rag_scorecard.md` |

**What you've built:** a full RAG pipeline — retrieve → assemble → generate → cite → evaluate.
`rag_generate(ticket, fields)` is the production function for Phase 5.

**Next:** Module 4.5 — GraphRAG (optional).
For queries that need connecting facts across documents, a knowledge graph helps.
Skip if top-k retrieval achieves hit-rate@3 ≥ 0.90 on your gold set.